# Florida outage recovery exploration

This notebook does exploratory plotting only.

It:
- loads a small Florida subset from `POUS.csv`
- pulls only the needed county and time window from `timeseries.pq`
- standardizes timestamps and FIPS codes
- joins weather on `event_id + geoid + datetime`
- makes a few dual-axis plots: `outageFraction` and `gust_mps`

It stops after plotting a small number of events.


In [ ]:
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.dataset as ds

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
plt.style.use('default')

POUS_PATH = Path(r'C:\Users\teaching\Downloads\outage-recovery-forecasting\data_raw\POUS.csv')
TS_PATH = Path(r'C:\Users\teaching\Downloads\outage-recovery-forecasting\data_raw\timeseries.pq')
WEATHER_PATH = Path(r'C:\Users\teaching\Downloads\outage-recovery-forecasting\data_raw\toy_tft_florida_event_weather_era5_hourly.csv')

FL_STATE_FIPS = '12'
STORM_HINT = '2017242N16333'
DATE_START = pd.Timestamp('2017-09-09')
DATE_END = pd.Timestamp('2017-09-13')
MAX_EVENTS_TO_PLOT = 4
PAD_BEFORE_HOURS = 24
PAD_AFTER_HOURS = 48


In [ ]:
def standardize_fips(series: pd.Series) -> pd.Series:
    return series.astype('string').str.strip().str.zfill(5)


def standardize_hourly_datetime(series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(series, errors='coerce')
    return dt.dt.floor('h')


def read_pous(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if 'CountyFIPS' not in df.columns:
        raise KeyError('POUS.csv must contain CountyFIPS as a column.')
    if 'event_start' not in df.columns:
        raise KeyError('POUS.csv must contain event_start as a column.')
    df = df.copy()
    df['CountyFIPS'] = standardize_fips(df['CountyFIPS'])
    df['event_start'] = pd.to_datetime(df['event_start'], errors='coerce')
    df['storm'] = df['storm'].astype('string')
    return df


def read_weather(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {'event_id', 'storm', 'geoid', 'datetime', 'event_start'}
    missing = required.difference(df.columns)
    if missing:
        raise KeyError(f'Weather file missing columns: {sorted(missing)}')
    df = df.copy()
    df['event_id'] = df['event_id'].astype('string')
    df['storm'] = df['storm'].astype('string')
    df['geoid'] = standardize_fips(df['geoid'])
    df['datetime'] = standardize_hourly_datetime(df['datetime'])
    df['event_start'] = standardize_hourly_datetime(df['event_start'])
    return df


def infer_event_id_for_row(row: pd.Series, weather: pd.DataFrame) -> pd.Series:
    exact = weather[
        (weather['storm'] == row['storm'])
        & (weather['event_start'] == row['event_start'].floor('h'))
        & (weather['geoid'] == row['CountyFIPS'])
    ]
    if exact.empty:
        exact = weather[
            (weather['storm'] == row['storm'])
            & (weather['event_start'] == row['event_start'].floor('h'))
        ]
    ids = exact['event_id'].dropna().unique()
    return ids[0] if len(ids) else pd.NA


def get_dataset_fields(dataset: ds.Dataset) -> tuple[str, str]:
    names = set(dataset.schema.names)
    date_field = None
    county_field = None
    for candidate in ['RecordDateTime', '__index_level_0__', 'datetime']:
        if candidate in names:
            date_field = candidate
            break
    for candidate in ['CountyFIPS', '__index_level_1__', 'geoid']:
        if candidate in names:
            county_field = candidate
            break
    if date_field is None or county_field is None:
        raise KeyError(f'Could not identify index fields in parquet schema. Available fields: {sorted(names)}')
    return date_field, county_field


def load_outage_window(
    pq_path: Path,
    county_fips: str,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
) -> pd.DataFrame:
    dataset = ds.dataset(pq_path, format='parquet')
    date_field, county_field = get_dataset_fields(dataset)

    county_type = dataset.schema.field(county_field).type
    county_value: object = int(county_fips) if pa.types.is_integer(county_type) else str(county_fips)

    filter_expr = (
        (ds.field(county_field) == county_value)
        & (ds.field(date_field) >= pd.Timestamp(window_start).to_pydatetime())
        & (ds.field(date_field) < pd.Timestamp(window_end).to_pydatetime())
    )

    table = dataset.to_table(
        columns=[date_field, county_field, 'OutageFraction', 'CustomersTracked'],
        filter=filter_expr,
    )
    df = table.to_pandas()
    if df.empty:
        return df

    df = df.rename(
        columns={
            date_field: 'datetime',
            county_field: 'CountyFIPS',
            'OutageFraction': 'outageFraction',
            'CustomersTracked': 'customersTracked',
        }
    )
    df['CountyFIPS'] = standardize_fips(df['CountyFIPS'])
    df['datetime'] = standardize_hourly_datetime(df['datetime'])
    return df.sort_values('datetime').reset_index(drop=True)


def merge_outage_weather(outage: pd.DataFrame, weather: pd.DataFrame, event_row: pd.Series) -> pd.DataFrame:
    if outage.empty:
        return outage
    event_id = event_row.get('event_id', pd.NA)
    if pd.isna(event_id):
        raise ValueError('event_id could not be inferred for the selected POUS row.')
    w = weather[
        (weather['event_id'] == event_id)
        & (weather['geoid'] == event_row['CountyFIPS'])
    ].copy()
    w = w.rename(columns={'geoid': 'CountyFIPS'})
    merged = outage.merge(w, on=['CountyFIPS', 'datetime'], how='inner', suffixes=('', '_weather'))
    return merged.sort_values('datetime').reset_index(drop=True)


def plot_dual_axis(df: pd.DataFrame, title: str) -> None:
    fig, ax1 = plt.subplots(figsize=(12, 4))
    ax2 = ax1.twinx()

    ax1.plot(df['datetime'], df['outageFraction'], label='outageFraction', linewidth=2)
    ax2.plot(df['datetime'], df['gust_mps'], label='gust_mps', linewidth=2)

    ax1.set_xlabel('datetime')
    ax1.set_ylabel('outageFraction')
    ax2.set_ylabel('gust_mps')
    ax1.set_title(title)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    ax1.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()


In [ ]:
pous = read_pous(POUS_PATH)
weather = read_weather(WEATHER_PATH)

florida = pous[
    pous['CountyFIPS'].str.startswith(FL_STATE_FIPS)
    & pous['event_start'].between(DATE_START, DATE_END)
].copy()

if florida.empty:
    raise ValueError('No Florida POUS rows found in the requested date window.')

if STORM_HINT in set(florida['storm'].dropna().astype(str)):
    florida = florida[florida['storm'] == STORM_HINT].copy()

florida = florida.sort_values(['event_start', 'CountyFIPS']).copy()
florida['event_id'] = florida.apply(infer_event_id_for_row, axis=1, weather=weather)

selected = florida[florida['event_id'].notna()].drop_duplicates(subset=['event_id', 'CountyFIPS']).head(MAX_EVENTS_TO_PLOT).copy()

selected[['event_start', 'storm', 'CountyFIPS', 'event_id', 'duration_hours', 'county_pop']]


In [ ]:
for _, event_row in selected.iterrows():
    event_start = pd.to_datetime(event_row['event_start'])
    duration_hours = pd.to_numeric(event_row.get('duration_hours'), errors='coerce')
    if pd.isna(duration_hours) or duration_hours <= 0:
        duration_hours = 24

    window_start = event_start - pd.Timedelta(hours=PAD_BEFORE_HOURS)
    window_end = event_start + pd.Timedelta(hours=max(float(duration_hours), 24) + PAD_AFTER_HOURS)

    outage = load_outage_window(TS_PATH, event_row['CountyFIPS'], window_start, window_end)
    if outage.empty:
        print(f"Skipping {event_row['CountyFIPS']} {event_row['storm']}: no outage rows in window.")
        continue

    merged = merge_outage_weather(outage, weather, event_row)
    if merged.empty:
        print(f"Skipping {event_row['CountyFIPS']} {event_row['storm']}: no merged weather rows.")
        continue

    title = (
        f"storm={event_row['storm']} | county={event_row['CountyFIPS']} | "
        f"event_id={event_row['event_id']} | start={event_start:%Y-%m-%d %H:%M}"
    )
    plot_dual_axis(merged, title)

print('Done. This notebook stops after a few exploratory plots.')
